# 05 — Genetic programming per β(t): esperimento sulla seconda ondata

**Impostazione** (decisioni dell'11/11/2026):
- finestra **1/9/2020 – 31/1/2021**, training sui primi 2/3 (fino all'11/12/2020), test sul resto;
- **β(t) = β_cost · exp(g(x(t)))**, con g l'albero evoluto e β_cost il β della baseline costante; β limitata a [10⁻⁴, 2];
- **tre configurazioni di terminali**: solo `t` (in mesi); solo mobilità (`retail`, `work` con ritardi 0, 3, 7, 10, 14); entrambi;
- σ fisso (**1/4** di riferimento, **1/3** come sensibilità; la prova pilota era con 1/5,2), γ(t) e μ(t) dai dati, **E₀ stimato per ogni formula** come nelle baseline;
- NSGA-II su (errore normalizzato sul training, numero di nodi), (μ + λ) con 200 + 200 individui, 100 generazioni, crossover oppure mutazione, profondità ≤ 6, duplicati eliminati, costanti del front ottimizzate con Nelder-Mead ogni 5 generazioni;
- **10 seed** per configurazione e per σ.

Codice in `src/seird_gp/gp.py`; run con `scripts/run_gp.py` (circa 7 minuti l'una, 60 run in circa 4 ore con 2 core). Qui si leggono i Pareto front salvati in `results/gp/`.

**Regola per scegliere una formula per run** (fissata prima di guardare i risultati): la formula più semplice del Pareto front con errore di training entro il 5% del migliore di quella run. Usa solo il training: il test resta un controllo onesto. La regola del "gomito" è stata scartata dopo la prova pilota: con front di 3–7 formule sceglieva formule banali.

In [ ]:
import glob, json, pickle, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from seird_gp.data import PROCESSED_DIR, RESULTS_DIR
from seird_gp.beta_models import load_restrictions

TOL = 0.05
SIGMA_KEYS = {"4": "σ = 1/4 (latenza)", "3": "σ = 1/3 (latenza, Europa)", "5.2": "σ = 1/5.2"}
CONFIG_ORDER = ["tempo", "mobilita", "tempo+mobilita"]

def load_runs(tag="full_"):
    rows = []
    for f in sorted(glob.glob(str(RESULTS_DIR / "gp" / f"{tag}*.json"))):
        m = re.search(r"_s([\d.]+)_(.+)_seed(\d+)\.json$", f)
        front = pd.DataFrame(json.load(open(f)))
        front["beta"] = front["beta"].apply(np.array)
        rows.append({"σ": m.group(1), "config": m.group(2), "seed": int(m.group(3)), "front": front})
    return pd.DataFrame(rows)

def choose(front, tol=TOL):
    ok = front[front["fitness_norm train"] <= (1 + tol) * front["fitness_norm train"].min()]
    return ok.sort_values(["nodi", "fitness_norm train"]).iloc[0]

runs = load_runs()
chosen = pd.DataFrame([{**{k: r[k] for k in ["σ", "config", "seed"]}, **choose(r["front"]).to_dict()}
                       for _, r in runs.iterrows()])
print(len(runs), "run;", runs.groupby(["σ", "config"]).size().to_dict())

## 1. Il GP contro le baseline

Mediana e intervallo interquartile sui 10 seed della formula scelta, accanto alle baseline dello stesso σ (notebook 04).

In [ ]:
base = pickle.load(open(RESULTS_DIR / "04_baseline_all.pkl", "rb"))
W = "set20–gen21"
rows = []
for s, key in SIGMA_KEYS.items():
    for m in ["costante", "a tratti", "lineare retail", "lineare work", "lineare retail+work"]:
        if (W, m, key) in base:
            r = base[(W, m, key)]
            rows.append({"σ": s, "modello": f"baseline {m}", "train": r.rmsd.loc["train", "fitness_norm"],
                         "test": r.rmsd.loc["test", "fitness_norm"], "test IQR": "", "nodi": ""})
    for c in CONFIG_ORDER:
        g = chosen[(chosen["σ"] == s) & (chosen["config"] == c)]
        if len(g):
            q = g["fitness_norm test"].quantile([.25, .75])
            rows.append({"σ": s, "modello": f"GP {c} (mediana su {len(g)} seed)",
                         "train": g["fitness_norm train"].median(), "test": g["fitness_norm test"].median(),
                         "test IQR": f"{q.iloc[0]:.2f}–{q.iloc[1]:.2f}", "nodi": int(g["nodi"].median())})
summary = pd.DataFrame(rows).set_index(["σ", "modello"])
summary.to_csv(RESULTS_DIR / "05_gp_vs_baseline.csv")
summary.round(3)

## 2. Le formule scelte

Una riga per run. `ritardi` = ritardi della mobilità usati dalla formula.

In [ ]:
def lags(terms):
    return sorted({int(t.split("lag")[1]) for t in terms if "lag" in t})

chosen["ritardi"] = chosen["terminali usati"].apply(lags)
chosen["usa t"] = chosen["terminali usati"].apply(lambda t: "t" in t)
cols = ["σ", "config", "seed", "nodi", "formula", "fitness_norm train", "fitness_norm test", "E0", "ritardi", "usa t"]
chosen.sort_values(["σ", "config", "seed"])[cols].round(3)

## 3. Quali ritardi della mobilità sceglie il GP

Criterio del notebook 03, fissato prima dei risultati: se il GP usa di solito ritardi di 0–3 giorni il modello è coerente; se usa di solito 7 giorni o più manca il ritardo di notifica. Si guarda la distribuzione sui seed, non una singola run.

In [ ]:
mob = chosen[chosen["config"] != "tempo"].copy()
mob["ritardo minimo"] = mob["ritardi"].apply(lambda l: min(l) if l else np.nan)
mob["ritardo massimo"] = mob["ritardi"].apply(lambda l: max(l) if l else np.nan)
counts = (mob.explode("ritardi").dropna(subset=["ritardi"])
          .groupby(["σ", "config", "ritardi"]).size().unstack(fill_value=0))
print("Formule che usano almeno un indice di mobilità:", int(mob["ritardi"].apply(bool).sum()), "su", len(mob))
print("Quota con tutti i ritardi ≤ 3 giorni:", round((mob["ritardo massimo"] <= 3).mean(), 2))
print("Quota con almeno un ritardo ≥ 7 giorni:", round((mob["ritardo massimo"] >= 7).mean(), 2))
counts

## 4. β(t) delle formule scelte

In [ ]:
data = pd.read_csv(PROCESSED_DIR / "trento_clean.csv", index_col=0, parse_dates=True)
idx = data.loc["2020-09-01":"2021-01-31"].index
restr = load_restrictions()
s_ref = "4" if (chosen["σ"] == "4").any() else chosen["σ"].iloc[0]
fig, ax = plt.subplots(len(CONFIG_ORDER), 1, figsize=(11, 9), sharex=True, sharey=True)
pw = base.get((W, "a tratti", SIGMA_KEYS[s_ref]))
for a, c in zip(ax, CONFIG_ORDER):
    for _, r in chosen[(chosen["σ"] == s_ref) & (chosen["config"] == c)].iterrows():
        a.plot(idx, r["beta"], color="#1f4e79", alpha=0.35, lw=1)
    if pw is not None:
        a.plot(pw.extra["beta"], color="#a33", lw=1.5, label="baseline a tratti")
    a.axvspan(idx[int(round(len(idx) * 2 / 3))], idx[-1], color="#f4e3c1", alpha=0.6)
    for d in restr.inizio[(restr.inizio > idx[0]) & (restr.inizio <= idx[-1])]:
        a.axvline(d, color="#a33", lw=0.5, ls="--")
    a.set_title(f"GP {c}, σ = 1/{s_ref}: formule scelte dei 10 seed (blu)", fontsize=9)
    a.set_ylabel("β(t)")
    a.spines[["top", "right"]].set_visible(False)
ax[0].legend(frameon=False, fontsize=8)
fig.tight_layout(); fig.savefig(RESULTS_DIR / "05_beta_gp.png", dpi=150)

## 5. Lettura

*(da scrivere dopo i risultati)*